In [ ]:
import pandas as pd
import os

parquet_path = "../data/processed/criteo-uplift-v2.1.parquet"

In [ ]:
# Make the Parquet file from the CSV once. Later runs just load the Parquet.
if not os.path.exists(parquet_path):
    df = pd.read_csv("../data/raw/criteo-uplift-v2.1.csv")

    print(f'Rows with missing values (dropped): {df.isna().any(axis=1).sum()} of {len(df)}')
    df = df.dropna()

    for i in range(12):
        df[f'f{i}'] = df[f'f{i}'].astype('float32')

    df['treatment'] = df['treatment'].astype('int16')
    df['visit'] = df['visit'].astype('int8')
    df['conversion'] = df['conversion'].astype('int8')
    df['exposure'] = df['exposure'].astype('int16')

    df.to_parquet(parquet_path, index=False)

In [ ]:
from sklearn.model_selection import train_test_split

# Split rows 60/20/20 into train/validation/test, with the same treatment and conversion mix in each.
# Save the split as a column in the Parquet, not as three files.
# 'dev' marks 1.5M train rows for quick experiments.
df = pd.read_parquet(parquet_path)
if 'split' not in df.columns:
    strata = df['treatment'] * 2 + df['conversion']
    train_idx, rest_idx = train_test_split(df.index, train_size=0.6, stratify=strata, random_state=42)
    val_idx, test_idx = train_test_split(rest_idx, train_size=0.5, stratify=strata[rest_idx], random_state=42)
    dev_idx, _ = train_test_split(train_idx, train_size=1_500_000, stratify=strata[train_idx], random_state=42)

    df['split'] = 'train'
    df.loc[val_idx, 'split'] = 'validation'
    df.loc[test_idx, 'split'] = 'test'
    df['split'] = df['split'].astype('category')

    df['dev'] = False
    df.loc[dev_idx, 'dev'] = True

    df.to_parquet(parquet_path, index=False)

In [ ]:
# Check that every split has the same treatment share and outcome rates.
# After this, keep test locked until the end: load data without it, e.g.
# pd.read_parquet(parquet_path, filters=[('split', '!=', 'test')])
check = df.groupby('split', observed=True).agg(
    rows=('treatment', 'size'),
    treatment_share=('treatment', 'mean'),
    visit_rate=('visit', 'mean'),
    conversion_rate=('conversion', 'mean'),
)
dev = df[df['dev']]
check.loc['dev (from train)'] = [len(dev), dev['treatment'].mean(), dev['visit'].mean(), dev['conversion'].mean()]
check['rows'] = check['rows'].astype(int)
check.insert(1, 'row_share', check['rows'] / len(df))
print(check.to_string(float_format='{:.6f}'.format))